```text
┌───────────────────────────────────────────────────────┐
│                  TOTAL TRAINING DATA                  │
└───────────────────────────────────────────────────────┘
                           │
            ┌──────────────┴──────────────┐
            │                             │
            ▼                             ▼
┌───────────────────────┐   ┌─────────────────────────┐
│ ACADEMIC CLONE        │   │ SYNTHETIC DATASET       │
│ DATASETS              │   │ (LLM rewrites of        │
│ (BigCloneBench,       │   │ platforms)              │
│ POJ-104, etc.)        │   │                         │
└───────────────────────┘   └─────────────────────────┘
            │                             │
            └──────────────┬──────────────┘
                           ▼
┌───────────────────────────────────────────────────────┐
│            FINE-TUNING PHASE (Stage 1)                │
│ • Train Semantic Model (CodeBERT/DeepSeek)            │
│ • 50/50 mix of both datasets                          │
│ • Learns human copy-pasting and AI re-phrasing        │
└───────────────────────────────────────────────────────┘
                           │
                           ▼
┌───────────────────────────────────────────────────────┐
│           META-LEARNING PHASE (Stage 2)               │
│ • Lexical + Syntactic + Semantic similarities         │
│ • Train XGBoost Meta-Classifier                       │
│ • Combined similarity-score features                  │
└───────────────────────────────────────────────────────┘
```

# Previewing the datasets

In [1]:
# ! pip install pandas pyarrow

In [2]:
# import pandas as pd

In [5]:
# print("Loading BigCloneBench dataframes...")
# df_bcb_train = pd.read_parquet("/content/drive/MyDrive/Code_Datasets/bigclonebench_train.parquet")
# df_bcb_validation = pd.read_parquet("/content/drive/MyDrive/Code_Datasets/bigclonebench_validation.parquet")
# df_bcb_test = pd.read_parquet("/content/drive/MyDrive/Code_Datasets/bigclonebench_test.parquet")

Loading BigCloneBench dataframes...


In [6]:
# print("Loading POJ-104 dataframes...")
# df_poj_train = pd.read_csv("/content/drive/MyDrive/Code_Datasets/poj104_train.csv")
# df_poj_validation = pd.read_csv("/content/drive/MyDrive/Code_Datasets/poj104_validation.csv")
# df_poj_test = pd.read_csv("/content/drive/MyDrive/Code_Datasets/poj104_test.csv")

Loading POJ-104 dataframes...


In [7]:
# # Print the shape (rows, columns) of each independent dataframe to verify
# print(f"df_bcb_train shape:      {df_bcb_train.shape}")
# print(f"df_bcb_validation shape: {df_bcb_validation.shape}")
# print(f"df_bcb_test shape:       {df_bcb_test.shape}\n")

# print(f"df_poj_train shape:      {df_poj_train.shape}")
# print(f"df_poj_validation shape: {df_poj_validation.shape}")
# print(f"df_poj_test shape:       {df_poj_test.shape}")

df_bcb_train shape:      (901028, 6)
df_bcb_validation shape: (415416, 6)
df_bcb_test shape:       (415416, 6)

df_poj_train shape:      (32500, 3)
df_poj_validation shape: (8500, 3)
df_poj_test shape:       (12000, 3)


In [11]:
# df_bcb_train.head(2)

,id,id1,id2,func1,func2,label
0,0,13988825,8660836,private void setNodekeyInJsonResponse(Stri...,"public void transform(String style, String...",False
1,1,80378,18548122,public static void test(String args[]) {\n...,private static String loadUrlToString(Stri...,True


In [15]:
# df_poj_train.head(1)['code']

,code
0,"int numcount=0;\r\nvoid divide(int num,int x)\..."


In [17]:
# grouped_poj = df_poj_train.groupby("label")

# problem_id = 8

# df_problem_1 = grouped_poj.get_group(problem_id)

# for index, row in df_problem_1.head(3).iterrows():
#     print(row["code"][:150] + "...\n")
#     print("-" * 50)

void pai(int a[],int m);
void shuru(int a[10],int b[10],int m,int n);
void shuchu(int c[20],int m,int n);
void hubing(int c[20],int a[10],int b[10]...

--------------------------------------------------
/*
 * ModularizedProgramming.cpp
 *
 *  Created on: 2012-11-23
 *      Author: Cui Zhaoxiong Class4 1200012931
 */
int a[200];
int b[100];
int...

--------------------------------------------------
int m,n,a[1000],b[1000],c[2000],s,t;
void input()
{
	scanf("%d %d",&m,&n);
	for(s=0;s<m;s++)
		scanf("%d",&a[s]);
	for(s=0;s<n;s++)
		scanf("%d...

--------------------------------------------------


# Generating AI Code from the base

In [1]:
!pip install -q ollama pandas tqdm pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00


In [2]:
import os
import sys
import random
import pandas as pd
from tqdm import tqdm
from google.colab import drive
import ollama


In [3]:
print("Mounting Google Drive...")
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/Code_Datasets/"

INPUT_POJ = os.path.join(DRIVE_DIR, "poj104_train.csv")
INPUT_BCB = os.path.join(DRIVE_DIR, "bigclonebench_train.parquet")

OUTPUT_POJ = os.path.join(DRIVE_DIR, "synthetic_poj104.csv")
OUTPUT_BCB = os.path.join(DRIVE_DIR, "synthetic_bcb_files.parquet")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
print("Using local Ollama with qwen2.5-coder model...")
# Ensure you have the model pulled via: ollama pull qwen2.5-coder
model_id = "qwen2.5-coder"


Initializing DeepSeek-Coder-7B-Instruct-v1.5 with 4-bit Quantization...


config.json:   0%|          | 0.00/621 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/22.5k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

In [14]:
import re

def clean_bpe_artifacts(raw_text):
    if not isinstance(raw_text, str):
        return raw_text
    clean_text = raw_text.replace("Ġ", " ").replace("Ċ", "\n")
    clean_text = re.sub(r'\n{3,}', '\n\n', clean_text)
    return clean_text.strip()

def generate_code_variant(prompt_text, language="cpp"):
    response = ollama.chat(model=model_id, messages=[
      {
        'role': 'user',
        'content': prompt_text,
      },
    ])

    generated_text = response['message']['content']

    extracted_code = generated_text
    if "```" in generated_text:
        try:
            extracted_code = generated_text.split(f"```{language}")[1].split("```")[0].strip()
        except IndexError:
            try:
                extracted_code = generated_text.split("```")[1].split("```")[0].strip()
            except IndexError:
                pass

    return extracted_code.strip()


In [15]:
def synthesize_poj104_colab(num_variants_per_problem=500):
    print("\n--- Running Colab Integrated POJ-104 Track ---")

    if not os.path.exists(INPUT_POJ):
        print(f"🔴 Error: Cannot find input file at {INPUT_POJ}.")
        return

    df_poj = pd.read_csv(INPUT_POJ)
    grouped = df_poj.groupby("label")

    if os.path.exists(OUTPUT_POJ):
        print("Found existing checkpoint! Resuming...")
        df_existing = pd.read_csv(OUTPUT_POJ)
        synthetic_records = df_existing.to_dict("records")
        counts = df_existing["label"].value_counts()
        completed_labels = set(counts[counts >= num_variants_per_problem].index)
        print(f"Skipping {len(completed_labels)} already finished categories.")
    else:
        print("Starting fresh task.")
        synthetic_records = []
        completed_labels = set()

    for label, group in tqdm(grouped, desc="Processing POJ Problems"):
        if label in completed_labels:
            continue

        existing_count = sum(1 for r in synthetic_records if r["label"] == label)
        variants_to_generate = num_variants_per_problem - existing_count

        if variants_to_generate <= 0:
            continue

        all_human_codes = group["code"].dropna().tolist()

        for i in range(variants_to_generate):
            selected_samples = random.sample(all_human_codes, k=min(3, len(all_human_codes)))

            concatenated_block = ""
            for idx, code_sample in enumerate(selected_samples):

                safe_sample = str(code_sample)[:1000]
                concatenated_block += f"\n--- Sample {idx+1} ---\n{safe_sample}\n"

            prompt = f"""
            You are an advanced AI Code Generator. Analyze the following cluster of C++ source codes written to solve the EXACT SAME algorithmic problem.

            Cluster of Human Reference Solutions:
            {concatenated_block}

            Write a brand new, fully functional C++ source code that solves this identical problem using a completely distinct programming architecture, naming convention, and code layout.

            Strict Operational Rules:
            1. DO NOT reuse specific variable names or line-by-line formatting patterns from the references.
            2. Fully refactor the logic.
            3. Ensure the code is self-contained and clean.
            4. Respond with NOTHING else except the code inside a single ```cpp ... ``` block.
            """

            ai_code = generate_code_variant(prompt, language="cpp")
            synthetic_records.append({"code": ai_code, "label": label})


            if len(synthetic_records) % 50 == 0:
                pd.DataFrame(synthetic_records).to_csv(OUTPUT_POJ, index=False)

        pd.DataFrame(synthetic_records).to_csv(OUTPUT_POJ, index=False)
        print(f"\n[Secured] Problem {label} completely generated.")

In [16]:
def synthesize_bcb_colab(num_files_needed=25000):
    print("\n--- Running Colab Integrated BigCloneBench Track ---")

    if not os.path.exists(INPUT_BCB):
        print(f"🔴 Error: Cannot find input file at {INPUT_BCB}.")
        return

    df_bcb = pd.read_parquet(INPUT_BCB)

    if os.path.exists(OUTPUT_BCB):
        print("Found existing Parquet checkpoint! Resuming...")
        df_existing = pd.read_parquet(OUTPUT_BCB)
        synthetic_records = df_existing.to_dict("records")
        start_index = len(synthetic_records)
        print(f"Resuming from index {start_index}...")
    else:
        print("Starting fresh task.")
        synthetic_records = []
        start_index = 0

    all_java_functions = df_bcb["func1"].dropna().unique().tolist()
    counter = start_index

    pbar = tqdm(total=num_files_needed, initial=start_index, desc="Synthesizing Java")
    while counter < num_files_needed:
        selected_samples = random.sample(all_java_functions, k=3)

        concatenated_block = ""
        for idx, func_sample in enumerate(selected_samples):
            safe_sample = str(func_sample)[:1000]
            concatenated_block += f"\n--- Sample {idx+1} ---\n{safe_sample}\n"

        prompt = f"""
        You are an Enterprise Java Code Synthesis Engine. Examine this cluster of production Java methods.

        Cluster of Reference Java Implementations:
        {concatenated_block}

        Extract the core backend business logic and write a completely brand new, highly robust, and distinct Java method that satisfies that identical functional objective.

        Strict Operational Rules:
        1. Fully rename all local parameters and logic routines.
        2. Do not mirror any single reference layout. Heavy refactoring is strictly required.
        3. Respond with NOTHING else except the code block inside a single ```java ... ``` block.
        """

        ai_code = generate_code_variant(prompt, language="java")

        synthetic_records.append({
            "func": ai_code,
            "functional_group": counter // 25
        })
        counter += 1
        pbar.update(1)

        if len(synthetic_records) % 100 == 0:
            pd.DataFrame(synthetic_records).to_parquet(OUTPUT_BCB, index=False)

    pd.DataFrame(synthetic_records).to_parquet(OUTPUT_BCB, index=False)
    pbar.close()
    print("BigCloneBench File Track Fully Completed and Secured!")

In [ ]:
if __name__ == "__main__":
    synthesize_poj104_colab(num_variants_per_problem=500)
    synthesize_bcb_colab(num_files_needed=25000)


--- Running Colab Integrated POJ-104 Track ---
Starting fresh task.


Processing POJ Problems:   0%|          | 0/65 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
